# Exercise 2: Computational Graph Implementation for TinyML Wake Word Detection

## MAIE 5532: Machine Learning Systems - Week 2

### Learning Objectives:
• Understand computational graph construction for reverse mode AD
• Implement forward pass with strategic value storage
• Implement backward pass for complete gradient computation
• Apply memory management techniques for embedded systems
• Connect theory to practical wake word detection systems

### 🎯 What is a Computational Graph?

A computational graph is the fundamental data structure underlying reverse mode automatic differentiation. Think of it as a **recipe that remembers every step** of your computation:

- **Nodes**: Represent values (inputs, parameters, intermediate results, outputs)
- **Edges**: Represent operations that transform values
- **Forward Pass**: Cook the recipe and store intermediate ingredients
- **Backward Pass**: Reverse the recipe to find how each ingredient affects the final dish

### Why This Matters for Wake Word Detection:
Every time your smart speaker recognizes "Hey Siri" or "OK Google", it's using neural networks trained with computational graphs. The gradients computed through these graphs enable the system to learn from millions of voice samples.

### Real-World Impact:
- **Privacy**: On-device learning means your voice never leaves your device
- **Personalization**: Models adapt to your specific voice patterns
- **Efficiency**: Optimized for microcontroller deployment
- **Federated Learning**: Contribute to global model improvement without sharing data

In [1]:
# Essential imports for computational graph implementation
import math
import numpy as np
from IPython.core.display import HTML

# Utility function for clean output display
def show(title, *pairs):
    """
    Pretty printer for displaying results in a structured format
    This makes our output more readable and professional
    """
    print(title)
    for k, v in pairs:
        print(f"  {k}: {v}")

print("🚀 Exercise 2: Computational Graph for Wake Word Detection")
print("=" * 60)
print()
print("✅ Imports completed successfully!")
print()
print("📝 Code Explanation:")
print("  • numpy: Efficient numerical operations for matrix math")
print("  • math: Basic mathematical functions")
print("  • show(): Custom formatter for clean result display")
print()
print("🧠 Computational Graph Concept:")
print("  A computational graph tracks every operation in your neural network")
print("  Like a detailed recipe that remembers every cooking step")
print("  Forward pass: follow recipe, backward pass: reverse engineer it")

🚀 Exercise 2: Computational Graph for Wake Word Detection

✅ Imports completed successfully!

📝 Code Explanation:
  • numpy: Efficient numerical operations for matrix math
  • math: Basic mathematical functions
  • show(): Custom formatter for clean result display

🧠 Computational Graph Concept:
  A computational graph tracks every operation in your neural network
  Like a detailed recipe that remembers every cooking step
  Forward pass: follow recipe, backward pass: reverse engineer it


## 🏗️ Building the Computational Graph Infrastructure

### Node Design Philosophy

Our `ComputationNode` class is the building block of our graph. Each node needs to store:

1. **Value**: The actual numerical result (forward pass)
2. **Gradient**: How much the final output changes w.r.t. this node (backward pass)  
3. **Dependencies**: Which nodes this node depends on (graph structure)
4. **Backward Function**: How to compute gradients for this specific operation

### Memory Management Strategy

For embedded systems, we need to be extremely careful about memory usage:
- **Pre-allocation**: Know memory requirements at compile time
- **Strategic Storage**: Only store what's absolutely necessary for backward pass
- **Memory Pools**: Reuse memory locations when possible
- **Mixed Precision**: Use 16-bit where possible, 32-bit where necessary

### The Magic of Reverse Mode

Unlike forward mode (which computes one input's derivative at a time), reverse mode:
- ✅ Computes ALL parameter gradients in one backward pass
- ✅ Efficient for neural networks (millions of parameters, one loss)
- ⚠️ Requires storing intermediate values (memory cost)
- 🎯 Perfect for training, challenging for inference-only embedded systems

In [2]:
class ComputationNode:
    """
    A node in our computational graph for reverse mode automatic differentiation
    
    This is like a 'smart container' that holds:
    - The value computed during forward pass
    - The gradient computed during backward pass  
    - References to parent nodes (dependencies)
    - A function to compute gradients (backward function)
    
    Think of it as a recipe step that remembers both the result
    and how to undo/reverse the step for gradient computation.
    """
    
    def __init__(self, value, name="", requires_grad=True):
        """
        Create a new computation node
        
        Args:
            value: The numerical value (scalar, vector, or matrix)
            name: Human-readable identifier for debugging
            requires_grad: Whether this node needs gradient computation
        """
        # Store value as numpy array for consistent operations
        self.value = np.array(value, dtype=np.float32)
        
        # Initialize gradient to zeros (will be filled during backward pass)
        self.gradient = np.zeros_like(self.value)
        
        # Metadata for debugging and graph visualization
        self.name = name
        self.requires_grad = requires_grad
        
        # Graph structure information
        self.backward_fn = None  # Function to compute gradients
        self.inputs = []        # Parent nodes in the graph
        
        print(f"📦 Created node '{self.name}':")
        print(f"    Shape: {self.value.shape}")
        print(f"    Value: {self.value}")
        print(f"    Requires grad: {requires_grad}")
    
    def __repr__(self):
        """String representation for easy debugging"""
        return f"Node('{self.name}', shape={self.value.shape}, grad_norm={np.linalg.norm(self.gradient):.6f})"

print("✅ ComputationNode class implemented!")
print()
print("🔍 What We Just Built:")
print("  • A 'smart container' for values and gradients")
print("  • Memory-efficient storage with numpy arrays")
print("  • Graph structure tracking with input references")
print("  • Debugging support with names and representations")
print()
print("💡 Key Insight:")
print("  Each node is like a recipe step that remembers both the result")
print("  and how to reverse the computation for gradient flow")

✅ ComputationNode class implemented!

🔍 What We Just Built:
  • A 'smart container' for values and gradients
  • Memory-efficient storage with numpy arrays
  • Graph structure tracking with input references
  • Debugging support with names and representations

💡 Key Insight:
  Each node is like a recipe step that remembers both the result
  and how to reverse the computation for gradient flow


## 🧮 Implementing Neural Network Operations

### Linear Layer: The Workhorse of Neural Networks

The linear (fully connected) layer is fundamental: **output = input @ weight.T + bias**

### (The @ symbol is the Python operator for matrix multiplication)

**Forward Pass Mathematics:**
- Matrix multiplication: combines input features using learned weights
- Bias addition: shifts the decision boundary
- Result: transformed representation for next layer

**Backward Pass Mathematics (Chain Rule):**
- ∂Loss/∂input = ∂Loss/∂output @ weight
- ∂Loss/∂weight = ∂Loss/∂output.T @ input  
- ∂Loss/∂bias = sum(∂Loss/∂output)

### Why This Implementation Strategy?

1. **Modularity**: Each operation is self-contained
2. **Composability**: Operations can be chained together
3. **Automatic Gradients**: Backward functions handle chain rule automatically
4. **Memory Efficiency**: Store only what's needed for backward pass

### Real-World Connection

This is exactly how PyTorch and TensorFlow implement their operations! Our implementation shows the core principles without the complexity of production frameworks.

In [3]:
def linear_layer(input_node, weight_node, bias_node, name="linear"):
    """
    Linear (fully connected) layer: output = input @ weight.T + bias
    
    This is the fundamental building block of neural networks.
    We compute the forward pass AND set up the backward pass function.
    
    Mathematical operations:
    1. Matrix multiplication: input @ weight.T
    2. Bias addition: result + bias
    3. Store everything needed for backward pass
    
    Args:
        input_node: Input features (shape: [input_dim])
        weight_node: Weight matrix (shape: [output_dim, input_dim])  
        bias_node: Bias vector (shape: [output_dim])
        name: Human-readable name for this layer
    
    Returns:
        output_node: Result of linear transformation
    """
    print(f"\n🔄 Computing {name} layer")
    print(f"    Input shape: {input_node.value.shape}")
    print(f"    Weight shape: {weight_node.value.shape}")
    print(f"    Bias shape: {bias_node.value.shape}")
    
    # === FORWARD PASS COMPUTATION ===
    # Step 1: Matrix multiplication (input @ weight.T)
    matmul_result = input_node.value @ weight_node.value.T
    print(f"    After matrix multiplication: {matmul_result.shape}")
    
    # Step 2: Add bias
    linear_output = matmul_result + bias_node.value
    print(f"    After bias addition: {linear_output.shape}")
    print(f"    Output values: {linear_output}")
    
    # Create output node
    output_node = ComputationNode(linear_output, name=f"{name}_output")
    
    # Store input references for backward pass
    output_node.inputs = [input_node, weight_node, bias_node]
    
    def backward():
        """
        Backward pass for linear layer - implements chain rule
        
        Given gradient flowing back (∂Loss/∂output), compute:
        - ∂Loss/∂input = ∂Loss/∂output @ weight
        - ∂Loss/∂weight = ∂Loss/∂output.T @ input
        - ∂Loss/∂bias = ∂Loss/∂output (sum if batched)
        
        This is the chain rule in action!
        """
        print(f"\n⬅️ Backward pass for {name}")
        print(f"    Incoming gradient shape: {output_node.gradient.shape}")
        print(f"    Incoming gradient: {output_node.gradient}")
        
        # Gradient w.r.t. input: ∂Loss/∂input = ∂Loss/∂output @ weight
        if input_node.requires_grad:
            input_grad = output_node.gradient @ weight_node.value
            input_node.gradient += input_grad
            print(f"    ∂Loss/∂input: {input_grad}")
        
        # Gradient w.r.t. weight: ∂Loss/∂weight = ∂Loss/∂output.T @ input
        if weight_node.requires_grad:
            # Reshape for proper matrix multiplication
            grad_reshaped = output_node.gradient.reshape(-1, 1) if output_node.gradient.ndim == 1 else output_node.gradient
            input_reshaped = input_node.value.reshape(1, -1) if input_node.value.ndim == 1 else input_node.value
            
            weight_grad = grad_reshaped @ input_reshaped
            weight_node.gradient += weight_grad
            print(f"    ∂Loss/∂weight shape: {weight_grad.shape}")
            print(f"    ∂Loss/∂weight: {weight_grad}")
        
        # Gradient w.r.t. bias: ∂Loss/∂bias = ∂Loss/∂output
        if bias_node.requires_grad:
            bias_grad = output_node.gradient.copy()
            bias_node.gradient += bias_grad
            print(f"    ∂Loss/∂bias: {bias_grad}")
    
    # Attach backward function to output node
    output_node.backward_fn = backward
    return output_node

print("✅ Linear layer implementation complete!")
print()
print("🔍 What This Function Does:")
print("  • Forward: Computes output = input @ weight.T + bias")
print("  • Backward: Sets up gradient computation using chain rule")
print("  • Memory: Stores references to inputs (not copies!)")
print("  • Modularity: Can be chained with other operations")
print()
print("💡 Chain Rule Implementation:")
print("  The backward function automatically applies calculus rules")
print("  No manual derivative computation required!")

✅ Linear layer implementation complete!

🔍 What This Function Does:
  • Forward: Computes output = input @ weight.T + bias
  • Backward: Sets up gradient computation using chain rule
  • Memory: Stores references to inputs (not copies!)
  • Modularity: Can be chained with other operations

💡 Chain Rule Implementation:
  The backward function automatically applies calculus rules
  No manual derivative computation required!


## 🎛️ Activation Functions: Non-linearity and Gradient Flow

### Tanh Activation: Smooth Non-linearity

**Mathematical Properties:**
- **Range**: (-1, 1) - keeps values bounded
- **Derivative**: 1 - tanh²(x) - easy to compute from forward pass value
- **Zero-centered**: Helps with gradient flow in deep networks

**Why Tanh for Hidden Layers?**
- Smooth gradients (no sudden jumps)
- Zero-centered outputs help next layer training
- Saturates gracefully (gradients approach 0 at extremes)

### ReLU: The Deep Learning Revolution

**Mathematical Properties:**
- **Function**: max(0, x)
- **Derivative**: 1 if x > 0, else 0
- **Computational**: Extremely efficient (just comparison)

**Why ReLU Changed Everything:**
- Solved vanishing gradient problem in deep networks
- Sparse activation (many neurons = 0)
- Computationally efficient for embedded systems

### The Vanishing Gradient Problem
This gradient is calculated by repeatedly multiplying many small numbers together (specifically, the derivatives of each layer's activation function, like sigmoid or tanh, which are often between 0 and 1).

Just like repeatedly multiplying fractions (e.g., 0.5 * 0.5 * 0.5...), this chain of multiplications causes the gradient to shrink exponentially with each step backward until it becomes vanishingly small and the early layers stop learning.

RelU transforms these small fractions to 1 (if they are > 0).

### Softmax: Probability Distribution

**Mathematical Properties:**
- **Function**: exp(xi) / sum(exp(x))
- **Output**: Valid probability distribution (sums to 1)
- **Derivative**: Complex but elegant when combined with cross-entropy loss

**Perfect for Classification:**
- Converts logits to probabilities
- Amplifies differences between classes
- Integrates beautifully with cross-entropy loss

In [4]:
def tanh_activation(input_node, name="tanh"):
    """
    Hyperbolic tangent activation function
    
    Forward: tanh(x) = (e^x - e^(-x)) / (e^x + e^(-x))
    Backward: d/dx[tanh(x)] = 1 - tanh²(x)
    
    Beautiful property: derivative can be computed from forward pass result!
    This saves computation and memory during backward pass.
    """
    print(f"\n🎛️ Computing {name} activation")
    print(f"    Input values: {input_node.value}")
    
    # Forward pass: compute tanh
    tanh_output = np.tanh(input_node.value)
    output_node = ComputationNode(tanh_output, name=f"{name}_output")
    output_node.inputs = [input_node]
    
    print(f"    Output values: {output_node.value}")
    print(f"    Output range: [{np.min(output_node.value):.3f}, {np.max(output_node.value):.3f}]")
    
    def backward():
        """
        Backward pass for tanh activation
        
        Key insight: tanh'(x) = 1 - tanh²(x)
        We can compute this from the forward pass result (output_node.value)!
        """
        print(f"\n⬅️ Backward pass for {name}")
        
        if input_node.requires_grad:
            # Derivative: 1 - tanh²(x)
            tanh_derivative = 1 - output_node.value ** 2
            
            # Apply chain rule: incoming_gradient * local_derivative
            input_grad = output_node.gradient * tanh_derivative
            input_node.gradient += input_grad
            
            print(f"    tanh derivative (1-tanh²): {tanh_derivative}")
            print(f"    Input gradient: {input_grad}")
    
    output_node.backward_fn = backward
    return output_node

def softmax_activation(input_node, name="softmax"):
    """
    Softmax activation for classification
    
    Forward: softmax(x)_i = exp(x_i) / sum(exp(x_j))
    Properties:
    - Outputs sum to 1 (valid probability distribution)
    - Amplifies differences between classes
    - Numerically stable with max subtraction trick
    """
    print(f"\n🎯 Computing {name} activation")
    print(f"    Input logits: {input_node.value}")
    
    # Numerical stability: subtract max to prevent overflow
    max_val = np.max(input_node.value)
    stable_input = input_node.value - max_val
    print(f"    After stability adjustment: {stable_input}")
    
    # Compute softmax
    exp_values = np.exp(stable_input)
    softmax_output = exp_values / np.sum(exp_values)
    
    output_node = ComputationNode(softmax_output, name=f"{name}_output")
    output_node.inputs = [input_node]
    
    print(f"    Output probabilities: {output_node.value}")
    print(f"    Probability sum: {np.sum(output_node.value):.6f} (should be 1.0)")
    
    def backward():
        """
        Backward pass for softmax
        
        The gradient is complex, but when combined with cross-entropy loss,
        it simplifies beautifully to: softmax_output - target
        """
        print(f"\n⬅️ Backward pass for {name}")
        
        if input_node.requires_grad:
            # Softmax Jacobian matrix computation
            s = output_node.value.reshape(-1, 1)
            jacobian = np.diagflat(s) - np.dot(s, s.T)
            
            # Apply chain rule
            input_grad = jacobian @ output_node.gradient.reshape(-1, 1)
            input_node.gradient += input_grad.flatten()
            
            print(f"    Jacobian shape: {jacobian.shape}")
            print(f"    Input gradient: {input_node.gradient}")
    
    output_node.backward_fn = backward
    return output_node

print("✅ Activation functions implemented!")
print()
print("🎛️ Activation Function Properties:")
print("  • tanh: Smooth, zero-centered, bounded (-1, 1)")
print("  • softmax: Probability distribution, perfect for classification")
print("  • Derivatives computed efficiently from forward pass values")
print()
print("🧠 Deep Learning Insight:")
print("  Activation functions provide non-linearity that enables")
print("  neural networks to learn complex patterns and decision boundaries")

✅ Activation functions implemented!

🎛️ Activation Function Properties:
  • tanh: Smooth, zero-centered, bounded (-1, 1)
  • softmax: Probability distribution, perfect for classification
  • Derivatives computed efficiently from forward pass values

🧠 Deep Learning Insight:
  Activation functions provide non-linearity that enables
  neural networks to learn complex patterns and decision boundaries


## 🎯 Loss Function: Cross-Entropy for Classification

### Why Cross-Entropy Loss?

Cross-entropy loss is the gold standard for classification because:

1. **Probabilistic Interpretation**: Measures how far predicted probabilities are from true distribution
2. **Gradient Properties**: Provides strong gradients when predictions are wrong
3. **Mathematical Elegance**: Combines beautifully with softmax activation
4. **Convex**: Has nice optimization properties (no local minima for linear models)

### The Mathematical Beauty

**Forward**: Loss = -∑(target_i × log(prediction_i))
**Backward** (with softmax): ∂Loss/∂logits = predictions - targets

This simplification is why softmax + cross-entropy is ubiquitous in deep learning!

### Numerical Stability Considerations

- **Log(0) problem**: Add small epsilon to prevent -∞
- **Overflow prevention**: Already handled by softmax stability
- **Gradient clipping**: Prevent exploding gradients in pathological cases

### Wake Word Detection Context

For wake word detection:
- **Class 0**: Wake word detected → target = [1, 0]
- **Class 1**: No wake word → target = [0, 1]
- **Training**: Minimize cross-entropy to improve classification accuracy

In [5]:
def cross_entropy_loss(predictions_node, target_node, name="cross_entropy"):
    """
    Cross-entropy loss for classification
    
    Forward: Loss = -sum(target * log(predictions))
    
    This is the standard loss function for classification tasks.
    It heavily penalizes confident wrong predictions and provides
    strong gradients for learning.
    
    Args:
        predictions_node: Softmax probabilities [num_classes]
        target_node: One-hot encoded true class [num_classes]
        
    Returns:
        loss_node: Scalar loss value
    """
    print(f"\n🎯 Computing {name} loss")
    print(f"    Predictions: {predictions_node.value}")
    print(f"    Target: {target_node.value}")
    
    # Numerical stability: prevent log(0)
    epsilon = 1e-15
    safe_predictions = np.clip(predictions_node.value, epsilon, 1 - epsilon)
    print(f"    Safe predictions (clipped): {safe_predictions}")
    
    # Cross-entropy computation: -sum(target * log(predictions))
    log_predictions = np.log(safe_predictions)
    loss_terms = target_node.value * log_predictions
    loss_value = -np.sum(loss_terms)
    
    print(f"    Log predictions: {log_predictions}")
    print(f"    Loss terms: {loss_terms}")
    print(f"    Final loss: {loss_value}")
    
    # Create scalar loss node
    loss_node = ComputationNode(loss_value, name=f"{name}_output")
    loss_node.inputs = [predictions_node, target_node]
    
    def backward():
        """
        Backward pass for cross-entropy loss
        
        Gradient: ∂Loss/∂predictions = -target / predictions
        
        When combined with softmax, this becomes simply:
        predictions - target (beautiful simplification!)
        """
        print(f"\n⬅️ Backward pass for {name}")
        
        if predictions_node.requires_grad:
            # Gradient of cross-entropy w.r.t. predictions
            pred_grad = -target_node.value / safe_predictions
            predictions_node.gradient += pred_grad
            
            print(f"    Gradient w.r.t. predictions: {pred_grad}")
            print(f"    Gradient interpretation:")
            for i, (pred, target, grad) in enumerate(zip(predictions_node.value, target_node.value, pred_grad)):
                if target > 0:  # True class
                    print(f"      Class {i} (TRUE): pred={pred:.4f}, grad={grad:.4f}")
                    print(f"        → {'Strong' if abs(grad) > 1 else 'Weak'} gradient pushes probability UP")
                else:  # False class  
                    print(f"      Class {i} (FALSE): pred={pred:.4f}, grad={grad:.4f}")
                    print(f"        → {'Strong' if abs(grad) > 1 else 'Weak'} gradient pushes probability DOWN")
    
    loss_node.backward_fn = backward
    return loss_node

print("✅ Cross-entropy loss implemented!")
print()
print("🎯 Loss Function Properties:")
print("  • Heavily penalizes confident wrong predictions")
print("  • Provides strong gradients when model is wrong")
print("  • Combines elegantly with softmax activation")
print("  • Numerically stable with epsilon clipping")
print()
print("💡 Training Insight:")
print("  The loss function guides learning by providing gradients")
print("  that push the model toward correct predictions")

✅ Cross-entropy loss implemented!

🎯 Loss Function Properties:
  • Heavily penalizes confident wrong predictions
  • Provides strong gradients when model is wrong
  • Combines elegantly with softmax activation
  • Numerically stable with epsilon clipping

💡 Training Insight:
  The loss function guides learning by providing gradients
  that push the model toward correct predictions


## 🎵 Wake Word Detection Network Implementation

### Network Architecture: 4 → 3 → 2

Our wake word detection network has a carefully designed architecture:

**Input Layer (4 features):**
- Spectral features (dominant frequency, spectral centroid)
- Energy features (RMS energy, zero-crossing rate)
- Temporal features (duration, pause detection)
- Amplitude features (peak amplitude, dynamic range)

**Hidden Layer (3 neurons):**
- Learns complex feature combinations
- Tanh activation for smooth gradients
- Enough capacity for simple wake word patterns

**Output Layer (2 classes):**
- Class 0: Wake word detected
- Class 1: No wake word (background/silence)
- Softmax activation for probability distribution

### Memory Analysis for Embedded Systems

This architecture is specifically designed for microcontroller deployment:
- **Total parameters**: (4×3 + 3) + (3×2 + 2) = 23 parameters
- **Memory for weights**: 23 × 4 bytes = 92 bytes
- **Memory for activations**: ~40 bytes during inference
- **Total inference memory**: <150 bytes (perfect for embedded!)

### Why This Size?

- **Computational constraints**: Must run in real-time on microcontroller
- **Memory constraints**: Fit in kilobytes of available RAM
- **Power constraints**: Minimize energy consumption for battery devices
- **Accuracy constraints**: Still need reasonable wake word detection performance

In [6]:
# Wake Word Detection Network Setup
print("🎵 WAKE WORD DETECTION NETWORK")
print("=" * 50)
print()
print("🏗️ Network Architecture: 4 → 3 → 2")
print("  • Input: 4 audio features")
print("  • Hidden: 3 neurons with tanh activation")
print("  • Output: 2 classes (wake_word, no_wake_word)")
print()

# Input data for testing
audio_features = [0.2, -0.1, 0.5, 0.3]  # Simulated audio features
target = [1, 0]  # Wake word detected (one-hot encoded)

print("📊 Test Data:")
print(f"  Audio features: {audio_features}")
print(f"    Feature 0: {audio_features[0]} (spectral centroid)")
print(f"    Feature 1: {audio_features[1]} (zero-crossing rate)")  
print(f"    Feature 2: {audio_features[2]} (RMS energy)")
print(f"    Feature 3: {audio_features[3]} (spectral rolloff)")
print()
print(f"  Target: {target}")
print(f"    Interpretation: Wake word detected (class 0)")
print()

# Create input and target nodes
print("🔧 Creating computational graph nodes...")
input_node = ComputationNode(audio_features, name="audio_input", requires_grad=False)
target_node = ComputationNode(target, name="target", requires_grad=False)

print("✅ Input nodes created!")
print(f"  Input node: {input_node}")
print(f"  Target node: {target_node}")

🎵 WAKE WORD DETECTION NETWORK

🏗️ Network Architecture: 4 → 3 → 2
  • Input: 4 audio features
  • Hidden: 3 neurons with tanh activation
  • Output: 2 classes (wake_word, no_wake_word)

📊 Test Data:
  Audio features: [0.2, -0.1, 0.5, 0.3]
    Feature 0: 0.2 (spectral centroid)
    Feature 1: -0.1 (zero-crossing rate)
    Feature 2: 0.5 (RMS energy)
    Feature 3: 0.3 (spectral rolloff)

  Target: [1, 0]
    Interpretation: Wake word detected (class 0)

🔧 Creating computational graph nodes...
📦 Created node 'audio_input':
    Shape: (4,)
    Value: [ 0.2 -0.1  0.5  0.3]
    Requires grad: False
📦 Created node 'target':
    Shape: (2,)
    Value: [1. 0.]
    Requires grad: False
✅ Input nodes created!
  Input node: Node('audio_input', shape=(4,), grad_norm=0.000000)
  Target node: Node('target', shape=(2,), grad_norm=0.000000)


In [7]:
# Layer 1: Audio features → Hidden layer (4 → 3)
print("\n🏗️ Building Layer 1: Audio Features → Hidden Layer")
print("-" * 55)

# Initialize weights and biases for first layer
# Small random weights prevent symmetry breaking issues
np.random.seed(42)  # For reproducible results
W1 = np.random.randn(3, 4).astype(np.float32) * 0.1  # Small initialization
b1 = np.zeros(3, dtype=np.float32)  # Start biases at zero

print("⚙️ Layer 1 Parameters:")
print(f"  Weight matrix W1 shape: {W1.shape}")
print(f"  W1 values:\n{W1}")
print(f"  Bias vector b1: {b1}")
print()

# Create parameter nodes
W1_node = ComputationNode(W1, name="W1", requires_grad=True)
b1_node = ComputationNode(b1, name="b1", requires_grad=True)

# Forward pass through first layer
print("🔄 Forward pass through Layer 1...")
z1_node = linear_layer(input_node, W1_node, b1_node, name="layer1_linear")
a1_node = tanh_activation(z1_node, name="layer1_activation")

print(f"✅ Layer 1 complete!")
print(f"  Pre-activation (z1): {z1_node.value}")
print(f"  Post-activation (a1): {a1_node.value}")
print(f"  Activation range: [{np.min(a1_node.value):.3f}, {np.max(a1_node.value):.3f}]")


🏗️ Building Layer 1: Audio Features → Hidden Layer
-------------------------------------------------------
⚙️ Layer 1 Parameters:
  Weight matrix W1 shape: (3, 4)
  W1 values:
[[ 0.04967142 -0.01382643  0.06476886  0.15230298]
 [-0.02341534 -0.0234137   0.15792128  0.07674348]
 [-0.04694744  0.054256   -0.04634177 -0.04657298]]
  Bias vector b1: [0. 0. 0.]

📦 Created node 'W1':
    Shape: (3, 4)
    Value: [[ 0.04967142 -0.01382643  0.06476886  0.15230298]
 [-0.02341534 -0.0234137   0.15792128  0.07674348]
 [-0.04694744  0.054256   -0.04634177 -0.04657298]]
    Requires grad: True
📦 Created node 'b1':
    Shape: (3,)
    Value: [0. 0. 0.]
    Requires grad: True
🔄 Forward pass through Layer 1...

🔄 Computing layer1_linear layer
    Input shape: (4,)
    Weight shape: (3, 4)
    Bias shape: (3,)
    After matrix multiplication: (3,)
    After bias addition: (3,)
    Output values: [ 0.08939224  0.09964199 -0.05195787]
📦 Created node 'layer1_linear_output':
    Shape: (3,)
    Value: [ 

In [8]:
# Layer 2: Hidden layer → Output layer (3 → 2)
print("\n🏗️ Building Layer 2: Hidden Layer → Output Classes")
print("-" * 55)

# Initialize weights and biases for second layer
W2 = np.random.randn(2, 3).astype(np.float32) * 0.1
b2 = np.zeros(2, dtype=np.float32)

print("⚙️ Layer 2 Parameters:")
print(f"  Weight matrix W2 shape: {W2.shape}")
print(f"  W2 values:\n{W2}")
print(f"  Bias vector b2: {b2}")
print()

# Create parameter nodes
W2_node = ComputationNode(W2, name="W2", requires_grad=True)
b2_node = ComputationNode(b2, name="b2", requires_grad=True)

# Forward pass through second layer
print("🔄 Forward pass through Layer 2...")
z2_node = linear_layer(a1_node, W2_node, b2_node, name="layer2_linear")
probs_node = softmax_activation(z2_node, name="layer2_softmax")

print(f"✅ Layer 2 complete!")
print(f"  Pre-softmax logits (z2): {z2_node.value}")
print(f"  Final probabilities: {probs_node.value}")
print(f"  Predicted class: {np.argmax(probs_node.value)} ({'Wake word' if np.argmax(probs_node.value) == 0 else 'No wake word'})")
print(f"  Confidence: {np.max(probs_node.value):.1%}")


🏗️ Building Layer 2: Hidden Layer → Output Classes
-------------------------------------------------------
⚙️ Layer 2 Parameters:
  Weight matrix W2 shape: (2, 3)
  W2 values:
[[ 0.02419623 -0.19132803 -0.17249179]
 [-0.05622875 -0.10128311  0.03142473]]
  Bias vector b2: [0. 0.]

📦 Created node 'W2':
    Shape: (2, 3)
    Value: [[ 0.02419623 -0.19132803 -0.17249179]
 [-0.05622875 -0.10128311  0.03142473]]
    Requires grad: True
📦 Created node 'b2':
    Shape: (2,)
    Value: [0. 0.]
    Requires grad: True
🔄 Forward pass through Layer 2...

🔄 Computing layer2_linear layer
    Input shape: (3,)
    Weight shape: (2, 3)
    Bias shape: (2,)
    After matrix multiplication: (2,)
    After bias addition: (2,)
    Output values: [-0.00789    -0.01670315]
📦 Created node 'layer2_linear_output':
    Shape: (2,)
    Value: [-0.00789    -0.01670315]
    Requires grad: True

🎯 Computing layer2_softmax activation
    Input logits: [-0.00789    -0.01670315]
    After stability adjustment: [ 0. 

In [9]:
# Compute loss and analyze results
print("\n🎯 Computing Loss and Network Performance")
print("-" * 50)

# Compute cross-entropy loss
loss_node = cross_entropy_loss(probs_node, target_node, name="training_loss")

print("\n📈 FORWARD PASS SUMMARY")
print("=" * 40)
show("Network Forward Pass Results",
     ("Input features", input_node.value),
     ("Hidden activations", a1_node.value),
     ("Output logits", z2_node.value),
     ("Output probabilities", probs_node.value),
     ("Training loss", f"{loss_node.value:.6f}"),
     ("Predicted class", np.argmax(probs_node.value)),
     ("True class", np.argmax(target_node.value)),
     ("Prediction correct?", np.argmax(probs_node.value) == np.argmax(target_node.value)))

# Network interpretation
print(f"\n🔍 Network Interpretation:")
print(f"  The network predicts class {np.argmax(probs_node.value)} with {np.max(probs_node.value):.1%} confidence")
print(f"  True class is {np.argmax(target_node.value)}")

if np.argmax(probs_node.value) == np.argmax(target_node.value):
    print(f"  ✅ Correct prediction! Loss = {loss_node.value:.6f}")
else:
    print(f"  ❌ Incorrect prediction. Loss = {loss_node.value:.6f}")
    print(f"  High loss indicates model needs more training")

print(f"\n💡 What the probabilities mean:")
for i, (prob, is_target) in enumerate(zip(probs_node.value, target_node.value)):
    class_name = "Wake word" if i == 0 else "No wake word"
    status = "✅ TRUE CLASS" if is_target else "❌ False class"
    print(f"  Class {i} ({class_name}): {prob:.1%} confidence {status}")


🎯 Computing Loss and Network Performance
--------------------------------------------------

🎯 Computing training_loss loss
    Predictions: [0.5022033  0.49779674]
    Target: [1. 0.]
    Safe predictions (clipped): [0.5022033  0.49779674]
    Log predictions: [-0.68875027 -0.6975634 ]
    Loss terms: [-0.68875027 -0.        ]
    Final loss: 0.6887502670288086
📦 Created node 'training_loss_output':
    Shape: ()
    Value: 0.6887502670288086
    Requires grad: True

📈 FORWARD PASS SUMMARY
Network Forward Pass Results
  Input features: [ 0.2 -0.1  0.5  0.3]
  Hidden activations: [ 0.08915489  0.09931352 -0.05191116]
  Output logits: [-0.00789    -0.01670315]
  Output probabilities: [0.5022033  0.49779674]
  Training loss: 0.688750
  Predicted class: 0
  True class: 0
  Prediction correct?: True

🔍 Network Interpretation:
  The network predicts class 0 with 50.2% confidence
  True class is 0
  ✅ Correct prediction! Loss = 0.688750

💡 What the probabilities mean:
  Class 0 (Wake word):

## 💾 Memory Analysis: Embedded System Constraints

### Memory Budget Analysis

For our 1KB constraint, we need to carefully account for every byte:

**Parameter Storage:**
- Weights: W1 (3×4) + W2 (2×3) = 18 float32 values = 72 bytes
- Biases: b1 (3) + b2 (2) = 5 float32 values = 20 bytes
- **Total parameters: 92 bytes**

**Activation Storage (Forward Pass):**
- Input: 4 float32 = 16 bytes
- Hidden: 3 float32 = 12 bytes  
- Output: 2 float32 = 8 bytes
- **Total activations: 36 bytes**

**Gradient Storage (Backward Pass):**
- Same as parameters: 92 bytes

**Scratch Space:**
- Temporary calculations: ~50 bytes

**Grand Total: ~270 bytes (well under 1KB!)**

### Memory Optimization Strategies

1. **Mixed Precision**: Use float16 for activations, float32 for gradients
2. **In-place Operations**: Reuse memory where possible
3. **Streaming**: Process one sample at a time (no batches)
4. **Quantization**: Use lower precision for inference

### Real-World Impact

This analysis shows that sophisticated neural networks CAN run on microcontrollers with careful memory management!

In [10]:
def analyze_memory_usage():
    """
    Comprehensive memory analysis for embedded deployment
    
    This function calculates exact memory requirements for our
    wake word detection network, ensuring it fits within
    embedded system constraints.
    """
    print("💾 COMPREHENSIVE MEMORY ANALYSIS")
    print("=" * 45)
    print()
    
    # Collect all nodes in our computational graph
    all_nodes = [
        input_node, target_node,
        W1_node, b1_node, z1_node, a1_node,
        W2_node, b2_node, z2_node, probs_node,
        loss_node
    ]
    
    print("📊 Node-by-Node Memory Breakdown:")
    print("Node Name           | Shape      | Elements | Value Bytes | Grad Bytes | Total")
    print("-" * 80)
    
    total_value_bytes = 0
    total_grad_bytes = 0
    
    for node in all_nodes:
        elements = node.value.size
        value_bytes = elements * 4  # float32 = 4 bytes
        grad_bytes = elements * 4 if node.requires_grad else 0
        total_bytes = value_bytes + grad_bytes
        
        total_value_bytes += value_bytes
        total_grad_bytes += grad_bytes
        
        print(f"{node.name:<18} | {str(node.value.shape):<10} | {elements:<8} | {value_bytes:<11} | {grad_bytes:<10} | {total_bytes}")
    
    print("-" * 80)
    print(f"{'TOTALS':<18} | {'':10} | {'':8} | {total_value_bytes:<11} | {total_grad_bytes:<10} | {total_value_bytes + total_grad_bytes}")
    
    # Category breakdown
    print(f"\n📈 Memory Category Breakdown:")
    
    # Parameters (weights and biases)
    param_nodes = [W1_node, b1_node, W2_node, b2_node]
    param_bytes = sum(node.value.nbytes + (node.gradient.nbytes if node.requires_grad else 0) for node in param_nodes)
    
    # Activations (intermediate values)
    activation_nodes = [input_node, z1_node, a1_node, z2_node, probs_node]
    activation_bytes = sum(node.value.nbytes for node in activation_nodes)
    
    # Loss and target
    other_bytes = target_node.value.nbytes + loss_node.value.nbytes
    
    print(f"  Parameters (weights + biases + gradients): {param_bytes} bytes")
    print(f"  Activations (forward pass storage): {activation_bytes} bytes")
    print(f"  Loss and targets: {other_bytes} bytes")
    print(f"  Total: {param_bytes + activation_bytes + other_bytes} bytes")
    
    # Constraint analysis
    constraint = 1024  # 1KB constraint
    total_used = total_value_bytes + total_grad_bytes
    
    print(f"\n🎯 Constraint Analysis:")
    print(f"  Memory budget: {constraint} bytes (1 KB)")
    print(f"  Memory used: {total_used} bytes")
    print(f"  Memory remaining: {constraint - total_used} bytes")
    print(f"  Memory efficiency: {(total_used/constraint)*100:.1f}% of budget used")
    
    if total_used <= constraint:
        print(f"  ✅ WITHIN BUDGET! System can deploy on embedded device")
    else:
        print(f"  ❌ EXCEEDS BUDGET! Need optimization for embedded deployment")
    
    # Optimization suggestions
    print(f"\n💡 Memory Optimization Opportunities:")
    if total_used <= constraint:
        print(f"  • Could use remaining {constraint - total_used} bytes for:")
        print(f"    - Larger network (more hidden units)")
        print(f"    - Input buffering for streaming audio")
        print(f"    - Multiple audio feature extractors")
    else:
        print(f"  • Use float16 for activations (50% reduction)")
        print(f"  • Quantize gradients to int16 (50% gradient memory reduction)")
        print(f"  • In-place operations to reuse activation memory")
    
    return total_used

# Run the memory analysis
memory_used = analyze_memory_usage()

💾 COMPREHENSIVE MEMORY ANALYSIS

📊 Node-by-Node Memory Breakdown:
Node Name           | Shape      | Elements | Value Bytes | Grad Bytes | Total
--------------------------------------------------------------------------------
audio_input        | (4,)       | 4        | 16          | 0          | 16
target             | (2,)       | 2        | 8           | 0          | 8
W1                 | (3, 4)     | 12       | 48          | 48         | 96
b1                 | (3,)       | 3        | 12          | 12         | 24
layer1_linear_output | (3,)       | 3        | 12          | 12         | 24
layer1_activation_output | (3,)       | 3        | 12          | 12         | 24
W2                 | (2, 3)     | 6        | 24          | 24         | 48
b2                 | (2,)       | 2        | 8           | 8          | 16
layer2_linear_output | (2,)       | 2        | 8           | 8          | 16
layer2_softmax_output | (2,)       | 2        | 8           | 8          | 16
training_los

## ⬅️ Reverse Mode Automatic Differentiation: The Backward Pass

### Understanding the Backward Pass

The backward pass is where the magic happens! Starting from the loss (a single scalar), we propagate gradients backward through the entire network to compute how much each parameter should change.

### The Backward Pass Algorithm

1. **Seed the Loss**: Set loss gradient to 1.0 (∂Loss/∂Loss = 1)
2. **Reverse Topological Order**: Visit nodes from output to input
3. **Apply Chain Rule**: Each node computes gradients for its inputs
4. **Accumulate Gradients**: Add contributions from multiple paths

### Why This Works

The chain rule from calculus: **∂Loss/∂param = ∂Loss/∂output × ∂output/∂param**

Each backward function implements the second term, and the backward pass automatically handles the first term through gradient propagation.

### Computational Efficiency

- **One backward pass** computes gradients for ALL parameters
- **Perfect for neural networks**: Many parameters, one loss
- **Scales beautifully**: Works for networks with millions of parameters

### Memory vs Computation Trade-off

- **Memory**: Must store intermediate values from forward pass
- **Computation**: Each operation computed once forward, once backward
- **Result**: ~2x computation, but enables training of complex models

In [11]:
print("⬅️ REVERSE MODE AUTOMATIC DIFFERENTIATION")
print("=" * 50)
print()
print("🌱 Starting backward pass from loss...")
print("This is where we compute gradients for ALL network parameters!")
print()

# Step 1: Seed the backward pass
print("STEP 1: Seeding the backward pass")
print("-" * 35)
loss_node.gradient = np.array(1.0, dtype=np.float32)
print(f"✅ Set loss gradient = {loss_node.gradient}")
print("   This means: ∂Loss/∂Loss = 1.0 (mathematical fact)")
print()

# Step 2: Define backward pass order (reverse topological order)
print("STEP 2: Defining backward pass execution order")
print("-" * 45)
backward_nodes = [loss_node, probs_node, z2_node, a1_node, z1_node]

print("📋 Backward pass execution order:")
for i, node in enumerate(backward_nodes):
    print(f"  {i+1}. {node.name} (flows gradients to its inputs)")
print()
print("💡 This order ensures gradients flow correctly through the graph")
print("   Each node receives gradients before computing gradients for its inputs")
print()

# Step 3: Execute backward pass
print("STEP 3: Executing backward pass")
print("-" * 35)
print("🔄 Computing gradients via reverse mode automatic differentiation...")
print()

for i, node in enumerate(backward_nodes):
    if node.backward_fn:
        print(f"🔄 Step {i+1}: Processing {node.name}")
        print(f"   Current gradient: {node.gradient}")
        node.backward_fn()
        print(f"   ✅ Gradients computed and propagated")
        print()

print("🎉 BACKWARD PASS COMPLETE!")
print("All parameter gradients have been computed automatically!")

⬅️ REVERSE MODE AUTOMATIC DIFFERENTIATION

🌱 Starting backward pass from loss...
This is where we compute gradients for ALL network parameters!

STEP 1: Seeding the backward pass
-----------------------------------
✅ Set loss gradient = 1.0
   This means: ∂Loss/∂Loss = 1.0 (mathematical fact)

STEP 2: Defining backward pass execution order
---------------------------------------------
📋 Backward pass execution order:
  1. training_loss_output (flows gradients to its inputs)
  2. layer2_softmax_output (flows gradients to its inputs)
  3. layer2_linear_output (flows gradients to its inputs)
  4. layer1_activation_output (flows gradients to its inputs)
  5. layer1_linear_output (flows gradients to its inputs)

💡 This order ensures gradients flow correctly through the graph
   Each node receives gradients before computing gradients for its inputs

STEP 3: Executing backward pass
-----------------------------------
🔄 Computing gradients via reverse mode automatic differentiation...

🔄 Step 

In [12]:
print("\n📊 GRADIENT ANALYSIS AND INTERPRETATION")
print("=" * 50)
print()

def analyze_gradients():
    """
    Analyze computed gradients to understand what the network learned
    and how parameters should be updated for better performance.
    """
    
    print("🔍 Parameter Gradient Summary:")
    print("-" * 40)
    
    # Collect all parameter gradients
    param_gradients = {
        'W1': W1_node.gradient,
        'b1': b1_node.gradient,
        'W2': W2_node.gradient,
        'b2': b2_node.gradient
    }
    
    for name, grad in param_gradients.items():
        grad_norm = np.linalg.norm(grad)
        grad_max = np.max(np.abs(grad))
        grad_mean = np.mean(grad)
        
        print(f"📈 {name} gradient:")
        print(f"    Shape: {grad.shape}")
        print(f"    Values: {grad}")
        print(f"    Norm (magnitude): {grad_norm:.6f}")
        print(f"    Max absolute: {grad_max:.6f}")
        print(f"    Mean: {grad_mean:.6f}")
        print()
    
    # Interpret gradient meanings
    print("🧠 Gradient Interpretation:")
    print("-" * 30)
    
    print("💡 What these gradients tell us:")
    print()
    
    # Analyze W2 gradients (output layer)
    print(f"🎯 Output Layer (W2) Analysis:")
    print(f"   W2 connects hidden layer to output classes")
    for i in range(W2_node.gradient.shape[0]):
        class_name = "Wake word" if i == 0 else "No wake word"
        row_grad = W2_node.gradient[i]
        direction = "increase" if np.mean(row_grad) > 0 else "decrease"
        strength = "strong" if np.linalg.norm(row_grad) > 0.1 else "weak"
        
        print(f"   Class {i} ({class_name}): {strength} {direction} signal")
        print(f"     Gradient: {row_grad}")
    print()
    
    # Analyze gradient flow strength
    total_grad_norm = sum(np.linalg.norm(grad) for grad in param_gradients.values())
    print(f"📊 Gradient Flow Analysis:")
    print(f"   Total gradient magnitude: {total_grad_norm:.6f}")
    
    if total_grad_norm > 1.0:
        print(f"   🚨 Large gradients detected - consider gradient clipping")
    elif total_grad_norm < 0.001:
        print(f"   ⚠️ Small gradients detected - learning might be slow")
    else:
        print(f"   ✅ Healthy gradient magnitudes for learning")
    
    return param_gradients

# Run gradient analysis
gradients = analyze_gradients()

# Show learning direction
print("🎓 Learning Direction Analysis:")
print("-" * 35)
print("If we apply these gradients (gradient descent), the network will:")

learning_rate = 0.01
print(f"With learning rate = {learning_rate}:")

for name, grad in gradients.items():
    update_magnitude = learning_rate * np.linalg.norm(grad)
    print(f"  {name}: Update magnitude = {update_magnitude:.6f}")
    
    if 'W' in name:  # Weight matrix
        print(f"    → Weight connections will be adjusted")
    else:  # Bias vector
        print(f"    → Bias terms will be shifted")

print()
print("💡 The network is learning to:")
print("   • Strengthen connections that help classify wake words correctly")
print("   • Weaken connections that lead to incorrect classifications")
print("   • Adjust biases to shift decision boundaries optimally")


📊 GRADIENT ANALYSIS AND INTERPRETATION

🔍 Parameter Gradient Summary:
----------------------------------------
📈 W1 gradient:
    Shape: (3, 4)
    Values: [[-0.00794341  0.00397171 -0.01985853 -0.01191512]
 [ 0.00887639 -0.0044382   0.02219098  0.01331459]
 [ 0.02024709 -0.01012354  0.05061771  0.03037063]]
    Norm (magnitude): 0.073351
    Max absolute: 0.050618
    Mean: 0.007943

📈 b1 gradient:
    Shape: (3,)
    Values: [-0.03971707  0.04438195  0.10123542]
    Norm (magnitude): 0.117456
    Max absolute: 0.101235
    Mean: 0.035300

📈 W2 gradient:
    Shape: (2, 3)
    Values: [[-0.04438101 -0.04943794  0.0258412 ]
 [ 0.04438101  0.04943794 -0.02584121]]
    Norm (magnitude): 0.100812
    Max absolute: 0.049438
    Mean: 0.000000

📈 b2 gradient:
    Shape: (2,)
    Values: [-0.49779668  0.4977967 ]
    Norm (magnitude): 0.703991
    Max absolute: 0.497797
    Mean: 0.000000

🧠 Gradient Interpretation:
------------------------------
💡 What these gradients tell us:

🎯 Output Lay

## 🎉 Exercise 2 Summary: Key Achievements

### What We Built

✅ **Complete Computational Graph Implementation**
- Node-based architecture for automatic differentiation
- Forward pass with strategic value storage
- Backward pass with automatic gradient computation

✅ **Realistic Wake Word Detection Network**
- 4 audio features → 3 hidden → 2 classes architecture
- Proper activation functions (tanh, softmax)
- Cross-entropy loss for classification

✅ **Memory-Efficient Design**
- Total memory usage: ~270 bytes (well under 1KB constraint)
- Strategic storage of only essential values
- Embedded-system-ready implementation

✅ **Educational Implementation**
- Step-by-step explanations of every operation
- Mathematical foundations clearly explained
- Real-world context and applications

### Key Insights Learned

🧠 **Computational Graphs Are Powerful**
- Enable automatic gradient computation for arbitrarily complex functions
- Form the backbone of all modern ML frameworks (PyTorch, TensorFlow)
- Perfect abstraction for implementing neural networks

🔄 **Reverse Mode AD Is Efficient**
- Computes ALL parameter gradients in one backward pass
- Scales perfectly with number of parameters
- Essential for training large neural networks

💾 **Memory Management Is Critical**
- Embedded systems require careful memory analysis
- Trade-offs between memory usage and computation
- Strategic checkpointing enables larger models

### Real-World Applications

This implementation demonstrates the core technology behind:
- **Smart speakers** (Alexa, Google Home, Siri)
- **Mobile voice assistants** 
- **Edge AI devices** with voice control
- **Federated learning** systems preserving privacy

### Next Steps

This computational graph forms the foundation for:
- Training larger, more sophisticated models
- Implementing advanced optimization algorithms
- Deploying on real embedded hardware
- Participating in federated learning networks